# Training: DeBERTa-v3-base (Optimized)
##### Ömer Faruk Merey - Middle East Technical University

Fine-tuning `microsoft/deberta-v3-base` with optimized hyperparameters from experiment sweep.

**Optimized config (based on experiment results):**
- **Epochs: 10** — best F1 at epoch 9, model benefits from longer training
- **Focal Loss (γ=2)** — 2nd best overall, better for class imbalance than weighted CE
- **Batch size: 4** — 3rd best, more gradient updates = better generalization on small data
- **Head+tail truncation (50/50)** — default split performed well
- LR=2e-5, weight_decay=0.01, warmup=0.1, dropout=0.1 (baseline values held)

All results logged to **wandb** project: `customer-sentiment-analysis`

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import matplotlib.pyplot as plt
import wandb
import gc

try:
    wandb.finish(quiet=True)
except:
    pass

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Load preprocessed data
train_df = pd.read_csv('../dataset/processed/train_split.csv')
val_df = pd.read_csv('../dataset/processed/val_split.csv')

with open('../dataset/processed/metadata.json') as f:
    metadata = json.load(f)

label_map = metadata['label_map']
label_map_inv = {v: k for k, v in label_map.items()}
class_weights = torch.tensor(metadata['class_weights'], dtype=torch.float32).to(device)

print(f"Train: {len(train_df)}, Val: {len(val_df)}")
print(f"Labels: {label_map}")
print(f"Class weights: {metadata['class_weights']}")

## 2. Tokenization with Head+Tail Truncation

Since ~50% of conversations exceed DeBERTa's 512 token limit, simple truncation loses the end of conversations where sentiment resolution often occurs.

**Strategy:** Take the first 254 tokens (problem statement) + last 255 tokens (resolution/outcome), joined with a  token.

In [ ]:
MODEL_NAME = 'microsoft/deberta-v3-base'
MAX_LENGTH = 512
BATCH_SIZE = 4
LEARNING_RATE = 2e-5
NUM_EPOCHS = 10
NUM_LABELS = 3
WARMUP_RATIO = 0.1

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Optimized config:")
print(f"  Epochs:     {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  LR:         {LEARNING_RATE}")
print(f"  Loss:       Focal Loss (γ=2)")

In [ ]:
def head_tail_tokenize(texts, tokenizer, max_length=MAX_LENGTH):
    """Tokenize with head+tail strategy for long texts."""
    all_input_ids = []
    all_attention_masks = []

    usable_tokens = max_length - 3  # [CLS] ... [SEP] ... [SEP]
    head_len = usable_tokens // 2
    tail_len = usable_tokens - head_len

    for text in texts:
        tokens = tokenizer.encode(text, add_special_tokens=False)

        if len(tokens) <= max_length - 2:
            encoding = tokenizer(
                text, max_length=max_length, padding='max_length',
                truncation=True, return_tensors='pt'
            )
        else:
            head = tokens[:head_len]
            tail = tokens[-tail_len:]
            combined = [tokenizer.cls_token_id] + head + [tokenizer.sep_token_id] + tail + [tokenizer.sep_token_id]
            assert len(combined) == max_length, f"Expected {max_length}, got {len(combined)}"
            encoding = {
                'input_ids': torch.tensor([combined]),
                'attention_mask': torch.tensor([[1] * max_length])
            }

        all_input_ids.append(encoding['input_ids'].squeeze(0))
        all_attention_masks.append(encoding['attention_mask'].squeeze(0))

    return {
        'input_ids': torch.stack(all_input_ids),
        'attention_mask': torch.stack(all_attention_masks)
    }

print("Tokenizing train set...")
train_encodings = head_tail_tokenize(train_df['conversation'].tolist(), tokenizer)
print("Tokenizing val set...")
val_encodings = head_tail_tokenize(val_df['conversation'].tolist(), tokenizer)

print(f"Train: {train_encodings['input_ids'].shape}")
print(f"Val:   {val_encodings['input_ids'].shape}")

## 3. Dataset & DataLoader

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

train_dataset = SentimentDataset(train_encodings, train_df['label'])
val_dataset = SentimentDataset(val_encodings, val_df['label'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 4. Model & Focal Loss Setup

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss — down-weights easy examples, focuses on hard ones."""
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, torch_dtype=torch.float32
)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

criterion = FocalLoss(weight=class_weights, gamma=2.0)

print(f"Model: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Total steps: {total_steps}, Warmup steps: {warmup_steps}")
print(f"Loss: FocalLoss(γ=2.0, weighted)")

## 5. Training Loop with W&B Logging

In [ ]:
wandb.init(
    project='customer-sentiment-analysis',
    name='deberta-v3-optimized',
    config={
        'model': MODEL_NAME,
        'max_length': MAX_LENGTH,
        'truncation': 'head+tail',
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'epochs': NUM_EPOCHS,
        'warmup_ratio': WARMUP_RATIO,
        'weight_decay': 0.01,
        'loss': 'focal_loss_gamma2',
        'class_weights': metadata['class_weights'],
        'train_size': len(train_df),
        'val_size': len(val_df),
        'optimized_from': 'experiment_sweep_top3'
    }
)

In [ ]:
def evaluate(model, loader, criterion, device):
    """Evaluate model on a data loader."""
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.float()
            loss = criterion(logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), np.array(all_preds), np.array(all_labels)

In [ ]:
best_val_f1 = 0
best_model_state = None

for epoch in range(NUM_EPOCHS):
    model.train()
    total_train_loss = 0

    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits.float(), labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_train_loss += loss.item()

        if (step + 1) % 20 == 0:
            wandb.log({
                'train/step_loss': loss.item(),
                'train/learning_rate': scheduler.get_last_lr()[0]
            })

    avg_train_loss = total_train_loss / len(train_loader)

    # Validation
    val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion, device)
    val_f1 = f1_score(val_labels, val_preds, average='weighted')
    val_acc = accuracy_score(val_labels, val_preds)
    val_f1_macro = f1_score(val_labels, val_preds, average='macro')
    val_f1_per = f1_score(val_labels, val_preds, average=None, zero_division=0)

    wandb.log({
        'epoch': epoch + 1,
        'train/loss': avg_train_loss,
        'val/loss': val_loss,
        'val/accuracy': val_acc,
        'val/f1_weighted': val_f1,
        'val/f1_macro': val_f1_macro,
        'val/f1_negative': val_f1_per[0],
        'val/f1_neutral': val_f1_per[1],
        'val/f1_positive': val_f1_per[2],
    })

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.4f} | "
          f"Val F1(w): {val_f1:.4f} | "
          f"Val F1(m): {val_f1_macro:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = model.state_dict().copy()
        print(f"  -> New best model (F1: {val_f1:.4f})")

print(f"
Best validation F1 (weighted): {best_val_f1:.4f}")

## 6. Training Analysis

In [ ]:
# Load best model for final evaluation
model.load_state_dict(best_model_state)

# Per-class F1 on validation
target_names = [label_map_inv[i] for i in range(NUM_LABELS)]
val_loss_final, val_preds_final, val_labels_final = evaluate(model, val_loader, criterion, device)
val_f1_per = f1_score(val_labels_final, val_preds_final, average=None, zero_division=0)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(NUM_LABELS)
bars = ax.bar(x, val_f1_per, color=['#e74c3c', '#3498db', '#2ecc71'])
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 Score — Validation Set (Optimized Model)')
ax.set_xticks(x)
ax.set_xticklabels(target_names)
ax.set_ylim(0, 1.1)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.2f}', ha='center', fontsize=12)
plt.tight_layout()
plt.show()

# Summary table
results_data = {
    'Metric': ['Accuracy', 'F1 (weighted)', 'F1 (macro)', 'F1 negative', 'F1 neutral', 'F1 positive'],
    'Validation': [
        f'{accuracy_score(val_labels_final, val_preds_final):.4f}',
        f'{f1_score(val_labels_final, val_preds_final, average="weighted"):.4f}',
        f'{f1_score(val_labels_final, val_preds_final, average="macro"):.4f}',
        f'{val_f1_per[0]:.4f}',
        f'{val_f1_per[1]:.4f}',
        f'{val_f1_per[2]:.4f}'
    ]
}
pd.DataFrame(results_data)

## 7. Save Model

In [ ]:
import os

save_dir = '../models/deberta-v3-sentiment'
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

config = {
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'label_map': label_map,
    'best_val_f1_weighted': best_val_f1,
    'training': {
        'epochs': NUM_EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'loss': 'focal_loss_gamma2',
        'weight_decay': 0.01,
        'warmup_ratio': WARMUP_RATIO,
        'truncation': 'head+tail_50/50'
    }
}
with open(f'{save_dir}/training_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f"Model saved to {save_dir}/")
for fname in sorted(os.listdir(save_dir)):
    size = os.path.getsize(f'{save_dir}/{fname}') / (1024*1024)
    print(f"  {fname}: {size:.1f} MB")

In [ ]:
wandb.finish()
print("Done.")